In [0]:
import os
import base64
import time
from datetime import datetime
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ObjectType, ExportFormat, ImportFormat
from pyspark.sql import Row

w = WorkspaceClient()
catalog = dbutils.widgets.get("catalog")
schemas_raw = dbutils.widgets.get("schemas")
schemas = [s.strip() for s in schemas_raw.split(",")]

for schema in schemas:

    switch_table = spark.sql(f"""
    SELECT table_name
    FROM {catalog}.information_schema.tables
    WHERE table_schema = '{schema}'
      AND table_name LIKE 'lakebridge_switch_%'
    ORDER BY table_name DESC
    LIMIT 1
    """).collect()[0]["table_name"]

    full_switch_table = f"{catalog}.{schema}.{switch_table}"
    # --- Drive from the Switch results table ---
    rows = spark.sql(f"""
        SELECT input_file_number, input_file_path, input_file_relative_path, export_output_path
        FROM {full_switch_table}
    """).collect()

    output_folder = f"/Workspace/Users/depoplimontj@delawareconsulting.com/delaware_lakebridge/switch_runs/{schema}"

    # --- Step 1: Replace placeholders ---
    for row in rows:
        # Derive workspace path from relative path
        notebook_path = os.path.join(
            output_folder,
            os.path.splitext(row["input_file_relative_path"])[0]
        )
        try:
            export_resp = w.workspace.export(notebook_path, format=ExportFormat.SOURCE)
            content = base64.b64decode(export_resp.content).decode("utf-8")
            content = content.replace("{catalog}", catalog).replace("{schema}", schema)
            obj = w.workspace.get_status(notebook_path)
            w.workspace.import_(
                notebook_path,
                content=base64.b64encode(content.encode("utf-8")).decode("utf-8"),
                format=ImportFormat.SOURCE,
                language=obj.language,
                overwrite=True,
            )
        except Exception as e:
            print(f"[WARN] Replace step failed for {notebook_path}: {e}")
            continue

    # --- Step 2: Run and track ---
    run_results = []

    for row in rows:
        notebook_path = os.path.join(
            output_folder,
            os.path.splitext(row["input_file_relative_path"])[0]
        )
        start = time.time()
        run_ts = datetime.now()

        try:
            dbutils.notebook.run(notebook_path, timeout_seconds=100000, arguments={
                "catalog": catalog,
                "schema": schema
            })
            run_results.append(Row(
                input_file_number=row["input_file_number"],
                input_file_path=row["input_file_path"],
                input_file_relative_path=row["input_file_relative_path"],
                notebook_path=notebook_path,
                run_status="SUCCESS",
                run_error=None,
                run_timestamp=run_ts,
                run_duration_seconds=round(time.time() - start, 2)
            ))
        except Exception as e:
            error_msg = str(e)
            status = "TIMEOUT" if "TimeoutException" in error_msg else "FAILED"
            run_results.append(Row(
                input_file_number=row["input_file_number"],
                input_file_path=row["input_file_path"],
                input_file_relative_path=row["input_file_relative_path"],
                notebook_path=notebook_path,
                run_status=status,
                run_error=error_msg[:2000],
                run_timestamp=run_ts,
                run_duration_seconds=round(time.time() - start, 2)
            ))
            print(f"[ERROR] {notebook_path}: {error_msg[:300]}")

    # --- Step 3: Write tracking results ---
    if run_results:
        spark.createDataFrame(run_results).write.mode("append").saveAsTable(
            f"{catalog}.{schema}.notebook_run_results"
        )
        print(f"[{schema}] {sum(1 for r in run_results if r.run_status == 'SUCCESS')}/{len(run_results)} notebooks succeeded")